# 08 — Verify Temporal Augmentation

Test each augmentation function from `train_balanced.py` to confirm it runs correctly.

In [10]:
import sys
import os
import numpy as np

sys.path.insert(0, os.path.join('..', 'scripts'))

import importlib
import train_balanced
importlib.reload(train_balanced)
from train_balanced import (add_spatial_jitter, time_warp, speed_variation,
                            frame_drop_repeat, augment_batch)

print("All augmentation functions imported successfully.")

All augmentation functions imported successfully.


## 1. Load Sample Data

In [11]:
DATA_DIR = os.path.join('..', 'data')
X_train = np.load(os.path.join(DATA_DIR, 'X_train.npy'))
print(f"Training data shape: {X_train.shape}")

batch = X_train[:4]
print(f"Test batch: {batch.shape}, dtype={batch.dtype}")
print(f"  Range: [{batch.min():.4f}, {batch.max():.4f}]")

Training data shape: (2965, 30, 132)
Test batch: (4, 30, 132), dtype=float64
  Range: [0.0004, 1.0000]


## 2. Test Spatial Jitter

In [12]:
jittered = add_spatial_jitter(batch, scale=0.01)
diff = jittered - batch
print(f"Input range:  [{batch.min():.4f}, {batch.max():.4f}]")
print(f"Output range: [{jittered.min():.4f}, {jittered.max():.4f}]")
print(f"Max diff:     {np.abs(diff).max():.6f}")
print(f"Shape preserved: {jittered.shape == batch.shape}")
print(f"Output clipped to [0,1]: {jittered.min() >= 0 and jittered.max() <= 1}")

Input range:  [0.0004, 1.0000]
Output range: [0.0004, 1.0000]
Max diff:     0.000000
Shape preserved: True
Output clipped to [0,1]: True


## 3. Test Time Warp

In [13]:
warped = time_warp(batch, max_stretch=0.2)
print(f"Input shape:  {batch.shape}")
print(f"Output shape: {warped.shape}")
print(f"Shape preserved: {warped.shape == batch.shape}")
print(f"Output range: [{warped.min():.4f}, {warped.max():.4f}]")
print(f"Mean abs diff: {np.abs(warped - batch).mean():.6f}")
diff_per_sample = np.abs(warped - batch).reshape(4, -1).mean(axis=1)
for i in range(4):
    print(f"  Sample {i}: mean abs diff = {diff_per_sample[i]:.6f}")

Input shape:  (4, 30, 132)
Output shape: (4, 30, 132)
Shape preserved: True
Output range: [0.0004, 1.0000]
Mean abs diff: 0.003024
  Sample 0: mean abs diff = 0.003152
  Sample 1: mean abs diff = 0.003287
  Sample 2: mean abs diff = 0.002875
  Sample 3: mean abs diff = 0.002781


## 4. Test Speed Variation

In [14]:
sped = speed_variation(batch, min_speed=0.8, max_speed=1.2)
print(f"Input shape:  {batch.shape}")
print(f"Output shape: {sped.shape}")
print(f"Shape preserved: {sped.shape == batch.shape}")
print(f"Output range: [{sped.min():.4f}, {sped.max():.4f}]")
print(f"Mean abs diff: {np.abs(sped - batch).mean():.6f}")
diff_per_sample = np.abs(sped - batch).reshape(4, -1).mean(axis=1)
for i in range(4):
    print(f"  Sample {i}: mean abs diff = {diff_per_sample[i]:.6f}")

Input shape:  (4, 30, 132)
Output shape: (4, 30, 132)
Shape preserved: True
Output range: [0.0004, 1.0000]
Mean abs diff: 0.012774
  Sample 0: mean abs diff = 0.013738
  Sample 1: mean abs diff = 0.014140
  Sample 2: mean abs diff = 0.012075
  Sample 3: mean abs diff = 0.011144


## 5. Test Frame Drop/Repeat

In [15]:
np.random.seed(42)
dropped = frame_drop_repeat(batch, drop_prob=0.15, repeat_prob=0.15)
print(f"Input shape:  {batch.shape}")
print(f"Output shape: {dropped.shape}")
print(f"Shape preserved: {dropped.shape == batch.shape}")
print(f"Output range: [{dropped.min():.4f}, {dropped.max():.4f}]")
print(f"Mean abs diff: {np.abs(dropped - batch).mean():.6f}")
changed = np.abs(dropped - batch).reshape(4, 30, -1).max(axis=2) > 1e-7
for i in range(4):
    n_changed = changed[i].sum()
    print(f"  Sample {i}: {n_changed}/30 frames modified")

Input shape:  (4, 30, 132)
Output shape: (4, 30, 132)
Shape preserved: True
Output range: [0.0004, 1.0000]
Mean abs diff: 0.003453
  Sample 0: 12/30 frames modified
  Sample 1: 10/30 frames modified
  Sample 2: 9/30 frames modified
  Sample 3: 10/30 frames modified


## 6. Test Combined Augmentation

In [16]:
np.random.seed(123)
augmented = augment_batch(batch)
print(f"Input shape:  {batch.shape}")
print(f"Output shape: {augmented.shape}")
print(f"Shape preserved: {augmented.shape == batch.shape}")
print(f"Input range:  [{batch.min():.4f}, {batch.max():.4f}]")
print(f"Output range: [{augmented.min():.4f}, {augmented.max():.4f}]")
print(f"Output clipped to [0,1]: {augmented.min() >= 0 and augmented.max() <= 1}")
print(f"Mean abs diff: {np.abs(augmented - batch).mean():.6f}")
print("All augmentations applied without errors: ✓")

Input shape:  (4, 30, 132)
Output shape: (4, 30, 132)
Shape preserved: True
Input range:  [0.0004, 1.0000]
Output range: [0.0004, 1.0000]
Output clipped to [0,1]: True
Mean abs diff: 0.009869
All augmentations applied without errors: ✓


## 7. Verify Reproducibility

In [17]:
np.random.seed(42)
r1 = augment_batch(batch)
np.random.seed(42)
r2 = augment_batch(batch)
print(f"Same seed gives same result: {np.allclose(r1, r2)}")

np.random.seed(999)
r3 = augment_batch(batch)
print(f"Different seed gives different result: {not np.allclose(r1, r3)}")

Same seed gives same result: False
Different seed gives different result: True
